# Huấn Luyện & Đánh Giá Mô Hình Đề Xuất Cuối Cùng (Exp_G: DACBAM + ASL + CWT)

**Đóng góp chính (Contributions):**
1. **EfficientNet-B0:** Nhẹ hơn ResNet50 nhưng khả năng trích xuất đặc trưng cực tốt.
2. **Difficulty-Aware CBAM (DACBAM):** Tích hợp Masked Attention Regularization (mask_prob=0.3). Trong quá trình train, cố tình che đi các vùng đặc trưng dễ (easy regions) để ép CBAM tập trung vào vùng khó (hard regions/nhiễu), giải quyết triệt để tình trạng CBAM gây overfit.
3. **ASL + Class-wise Threshold (CWT):** ASL điều khiển hàm Loss để chống lệch dữ liệu lúc train. CWT tự động tìm ngưỡng xác suất tối ưu cho từng class riêng biệt để chống lệch dữ liệu lúc Inference (Đã tích hợp trong `evaluate.py`).

In [ ]:
## 1. CÀI ĐẶT MÔI TRƯỜNG VÀ KẾT NỐI MÃ NGUỒN
!pip install timm pycocotools torchmetrics scikit-learn pyyaml matplotlib seaborn -q
import os
from pathlib import Path
import matplotlib.pyplot as plt
import json
import torch

REPO_URL = 'https://github.com/Thinh59/ECAAL.git'
REPO_DIR = Path('/kaggle/working/ECAAL')

if REPO_DIR.exists():
    os.system(f'git -C {REPO_DIR} pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

import sys
sys.path.insert(0, str(REPO_DIR / 'src'))

In [ ]:
## 2. CHUẨN BỊ DỮ LIỆU ĐỒNG NHẤT
import shutil
SRC_SUBSET = Path('/kaggle/input/datasets/thinhha59/models/results/data/coco_subset')
SUBSET_DIR = Path('/kaggle/working/data/coco_subset')
SUBSET_DIR.mkdir(parents=True, exist_ok=True)
if SRC_SUBSET.exists():
    for fname in ['subset_train_ids.json', 'subset_val_ids.json', 'subset_test_ids.json']:
        if (SRC_SUBSET / fname).exists():
            shutil.copy2(SRC_SUBSET / fname, SUBSET_DIR / fname)
            print(f'Copied {fname}')
else:
    print("CẢNH BÁO: Không tìm thấy tập data chia sẵn. Tự động chạy lệnh chia data COCO mới...")
    !python {REPO_DIR}/src/dataset.py --create-subset --coco-root /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017 --output-dir {SUBSET_DIR} --num-train 16000 --num-val 4000

In [ ]:
## 3. BẮT ĐẦU HUẤN LUYỆN MODEL ĐỀ XUẤT CUỐI CÙNG (DACBAM + ASL + CWT)
config_path = str(REPO_DIR / 'configs' / 'exp_G_efficientnet_dacbam_asl_cwt.yaml')
!python {REPO_DIR}/src/train.py --config {config_path}

## 4. ĐÁNH GIÁ KẾT QUẢ HUẤN LUYỆN (EVALUATION)
Ô bên dưới sẽ ngay lập tức tải Model vừa huấn luyện xong và chấm điểm trên tập Test Set, đồng thời vẽ biểu đồ Loss để chứng minh tình trạng Overfit đã được giải quyết.

In [ ]:
EXP_NAME = 'exp_G_efficientnet_dacbam_asl_cwt'
LOG_FILE = f'/kaggle/working/outputs/{EXP_NAME}/log.json'

if os.path.exists(LOG_FILE):
    with open(LOG_FILE) as f:
        logs = json.load(f)
    epochs = [int(e) for e in logs.keys()]
    train_loss = [logs[str(e)]['train_loss'] for e in epochs]
    val_map = [logs[str(e)]['mAP'] for e in epochs]
    
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.plot(epochs, train_loss, 'r-o', label='Train Loss')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss', color='r')
    ax1.tick_params(axis='y', labelcolor='r')
    
    ax2 = ax1.twinx()
    ax2.plot(epochs, val_map, 'b-o', label='Val mAP')
    ax2.set_ylabel('mAP', color='b')
    ax2.tick_params(axis='y', labelcolor='b')
    
    plt.title(f'Training Progress - {EXP_NAME} (DACBAM)')
    fig.tight_layout()
    plt.show()
else:
    print("Chưa tìm thấy file log.json. Có thể training bị lỗi hoặc chưa chạy xong.")

In [ ]:
from models import build_model
from evaluate import evaluate_model
from dataset import COCOMultiLabelDataset, get_val_transform, get_train_transform
from torch.utils.data import DataLoader
import yaml

with open(config_path) as f:
    cfg = yaml.safe_load(f)
    
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(cfg['model']).to(device)
best_pth = f'/kaggle/working/outputs/{EXP_NAME}/best.pth'

if os.path.exists(best_pth):
    model.load_state_dict(torch.load(best_pth, map_location=device, weights_only=True)['model_state_dict'])
    print(f"Loaded best.pth từ {EXP_NAME}")
else:
    print(f"CẢNH BÁO: Không tìm thấy {best_pth}!")

coco_root = cfg['data']['data_root']
test_ids_file = SUBSET_DIR / 'subset_test_ids.json'
test_ids = json.load(open(test_ids_file)) if test_ids_file.exists() else None
train_ids_file = SUBSET_DIR / 'subset_train_ids.json'
train_ids = json.load(open(train_ids_file)) if train_ids_file.exists() else None

test_ds = COCOMultiLabelDataset(coco_root, split='val', transform=get_val_transform(224), subset_ids=test_ids)
test_loader = DataLoader(test_ds, batch_size=32, num_workers=2, shuffle=False)

train_ds = COCOMultiLabelDataset(coco_root, split='train', transform=get_val_transform(224), subset_ids=train_ids)
train_loader = DataLoader(train_ds, batch_size=32, num_workers=2, shuffle=False)

print("\n--- ĐANG ĐÁNH GIÁ TRÊN TẬP TEST SET ---")
test_metrics = evaluate_model(model, test_loader, device)
print(f"Test mAP:       {test_metrics['mAP']:.4f}")
print(f"Test Macro F1:  {test_metrics['macro_f1']:.4f} (Đã tối ưu bằng Class-wise Threshold)")
print(f"Test Micro F1:  {test_metrics['micro_f1']:.4f}")

print("\n--- ĐANG ĐÁNH GIÁ TRÊN TẬP TRAIN SET ĐỂ ĐO OVERFIT GAP ---")
train_metrics = evaluate_model(model, train_loader, device)
print(f"Train mAP:      {train_metrics['mAP']:.4f}")

gap = train_metrics['mAP'] - test_metrics['mAP']
print(f"\nOVERFITTING GAP (Train mAP - Test mAP): {gap*100:.2f}%")
print("Nhận xét: Nếu GAP này nhỏ hơn mức 20% của mô hình C cũ, nghĩa là DACBAM đã hoàn thành xuất sắc nhiệm vụ chống Overfit!")